# 3. Score and compare methods

This notebook scores the predictions of the six benchmarked methods (as
deposited in the archive) and, if notebook 2 was run, your own ecCount run,
against the gold standard on the 12 image sets. It uses the benchmark's
object-matching rules, from `ecdna_bench.evaluation`.

**These numbers are illustrative.** Twelve images chosen to span the count
range are not a representative sample; the paper's values come from all 175
test images (or all 1,145 benchmark images). To reproduce those, follow route B
in `docs/TUTORIAL_EXTERNAL.md`.

**Needs:** notebook 1 run first.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ecdna_bench

# All files of the tutorials live here (next to the notebook by default).
DATA = Path(os.environ.get("ECDNA_TUTORIAL_DATA", "tutorial_data")).expanduser().resolve()
print("ecdna_bench :", getattr(ecdna_bench, "__version__", "?"), "from", Path(ecdna_bench.__file__).parent)
print("data folder :", DATA)

import cv2
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)   # hide notes about extra TIFF tags

def read_rgb(path):
    """RGB image as uint8 (H, W, 3), read the same way as the benchmark."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_gray(path):
    """Single-channel image (TIFF or PNG); colour images are reduced by their maximum."""
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    img = np.asarray(img)
    return img.max(axis=2) if img.ndim == 3 else img

def load_sample():
    """The table written by notebook 1."""
    table = DATA / "sample.csv"
    if not table.is_file():
        raise FileNotFoundError(f"{table} not found. Run notebook 1 first "
                                "(or set ECDNA_TUTORIAL_DATA to its data folder).")
    return pd.read_csv(table)

sample = load_sample()
pred_table = pd.read_csv(DATA / "predictions.csv")
METHODS = ["ecCount (peaks)", "ecCount (threshold mask)", "Label Engine", "MIA",
           "Classic (after opt)", "ecSeg"]
paths = {m: pred_table[pred_table["method"] == m].set_index("uid")["path"].map(lambda p: DATA / p)
         for m in METHODS}
own = DATA / "my_eccount" / "peaks"
if own.is_dir() and all((own / f"{u}.png").is_file() for u in sample["uid"]):
    METHODS.append("ecCount (peaks), this run")
    paths[METHODS[-1]] = pd.Series({u: own / f"{u}.png" for u in sample["uid"]})
print("methods:", ", ".join(METHODS))

## From deposited files to binary masks

Each method's output is turned into a binary mask with the benchmark's rules:
any non-zero pixel is foreground, except for Label Engine, whose deposited
output is its raw RGB rendering; it is converted to grey, kept above 0.5, and
8-connected components smaller than 3 px are removed.

In [ ]:
def prediction_mask(method, path):
    if method == "Label Engine":
        img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.shape[2] == 3 else img[:, :, 0]
        fg = (img.astype(np.float32) > 0.5).astype(np.uint8)
        n, lab, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
        keep = np.zeros(n, dtype=bool)
        keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= 3
        return keep[lab]
    return read_gray(path) > 0

## Matching predictions to the gold standard

Objects are 8-connected components of at least 3 px. A predicted object and a
gold-standard object form a candidate pair if their centroids are at most
20 px apart **or** their intersection over union is at least 0.1 (the OR
policy). The Hungarian algorithm then pairs objects one to one, minimising
`0.5 × (1 − IoU) + 0.5 × distance / 20`. Paired objects are true positives,
unpaired gold-standard objects false negatives, and unpaired predictions false
positives, except predictions that had a candidate but lost it to another
prediction: those are *ignored* and do not count as false positives.

In [ ]:
from ecdna_bench.evaluation import (objects_from_mask, precompute_pairwise,
                                    resolve_matching_from_pairwise, object_metrics_from_counts)

from skimage.measure import label, regionprops

D_MAX, IOU_MIN, ALPHA = 20.0, 0.1, 0.5

class ObjectMask:
    """One object's mask, read from the label image on demand. It gives the same
    pixels as a full-frame mask per object but needs far less memory on images
    with hundreds of ecDNA."""
    def __init__(self, labels, value):
        self.labels, self.value, self.shape = labels, value, labels.shape
    def __getitem__(self, index):
        return (self.labels[index] == self.value).astype(np.uint8)

def objects(mask):
    """8-connected components of at least 3 px, as the benchmark extracts them."""
    mask = np.asarray(mask) > 0
    objs = objects_from_mask(mask, min_area=3, connectivity=8, attach_mask=False)
    labels = label(mask, connectivity=2)
    values = [p.label for p in regionprops(labels) if p.area >= 3]
    assert len(values) == len(objs)
    for obj, value in zip(objs, values):
        obj["mask"] = ObjectMask(labels, value)
    return objs

def score(gt_mask, pred_mask):
    gt_objs = objects(gt_mask)
    pred_objs = objects(pred_mask)
    pw = precompute_pairwise(pred_objs, gt_objs, max_precompute_dist=D_MAX)
    m = resolve_matching_from_pairwise(pw, d_max=D_MAX, min_iou=IOU_MIN, alpha=ALPHA, policy="OR")
    return {"tp": m.tp, "fp": m.fp, "fn": m.fn, "ignored": m.ignored,
            "gt_count": len(gt_objs), "pred_count": len(pred_objs),
            "f1": object_metrics_from_counts(m.tp, m.fp, m.fn)["f1"]}

rows = []
for r in sample.itertuples():
    gt_mask = read_gray(DATA / r.gt)
    for method in METHODS:
        pred = prediction_mask(method, paths[method][r.uid])
        assert pred.shape == gt_mask.shape, (method, r.uid, pred.shape, gt_mask.shape)
        rows.append({"uid": r.uid, "cell_line": r.cell_line, "method": method, **score(gt_mask, pred)})
    print(f"  {r.uid} scored", flush=True)
per_image = pd.DataFrame(rows)
per_image.to_csv(DATA / "scores_per_image.csv", index=False)
per_image.query("uid == 'ncih2170_facs_fish_0723_low_her2_52'").drop(columns=["uid", "cell_line"])

## Pooled over the 12 images

As in the paper, object F1 is computed from the summed true positives, false
positives and false negatives of all images, and count error is the predicted
minus the gold-standard count per image (MAE: mean absolute error; bias: mean
signed error).

In [ ]:
per_image["count_error"] = per_image["pred_count"] - per_image["gt_count"]
pooled = per_image.groupby("method", sort=False).agg(
    tp=("tp", "sum"), fp=("fp", "sum"), fn=("fn", "sum"),
    count_mae=("count_error", lambda e: e.abs().mean()), count_bias=("count_error", "mean"))
pooled["object_f1"] = 2 * pooled.tp / (2 * pooled.tp + pooled.fp + pooled.fn)
pooled = pooled.loc[METHODS, ["tp", "fp", "fn", "object_f1", "count_mae", "count_bias"]]
pooled.to_csv(DATA / "scores_pooled.csv")
pooled.round({"object_f1": 3, "count_mae": 1, "count_bias": 1})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(pooled.index[::-1], pooled["object_f1"][::-1], color="steelblue")
ax[0].set_xlim(0, 1); ax[0].set_xlabel("object F1 (12 images, pooled)")
for y, v in enumerate(pooled["object_f1"][::-1]):
    ax[0].text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=9)
for method in METHODS:
    g = per_image[per_image["method"] == method]
    ax[1].scatter(g["gt_count"], g["pred_count"], s=25, label=method)
lim = [0, per_image[["gt_count", "pred_count"]].to_numpy().max() * 1.05]
ax[1].plot(lim, lim, "k--", lw=0.8); ax[1].set_xlim(lim); ax[1].set_ylim(lim)
ax[1].set_xlabel("gold-standard count"); ax[1].set_ylabel("predicted count"); ax[1].legend(fontsize=8)
plt.suptitle("Illustrative comparison on 12 test images (not the paper's values)")
plt.tight_layout(); plt.show()

## Next

* Reproduce the paper's tables on the 175 test images or all 1,145 benchmark
  images: route B in `docs/TUTORIAL_EXTERNAL.md` (download with
  `scripts/fetch_bia_subset.py`, prepare with `scripts/prepare_local_run.py`,
  score with `python -m ecdna_bench.cli.benchmark`).
* Score your own method: write one binary mask per image (same size as the
  gold standard) and pass it through `prediction_mask` and `score` above.